In [24]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import statsmodels.formula.api as smf

In [25]:
all_median_loss_results = []

In [26]:
def load_and_prepare_run(
    run_id,
    result_dir=""
):
    """
    Load loss, label, FOIF and TracIn results for one experimental run.

    Expected filenames:
        IF_sp1_de01_seed_<run_id>.csv
        TC_sp1_de01_seed_<run_id>.csv
        loss_sp1_de01_seed_<run_id>.csv
        label_sp1_de01_seed_<run_id>.csv
    """

    foif_df = pd.read_csv(
        f"IF_sp3_de001_run{run_id}.csv"
    )
    tracin_df = pd.read_csv(
        f"TC_sp3_de001_run{run_id}.csv"
    )
    loss_df = pd.read_csv(
        f"loss_sp3_de001_run{run_id}.csv"
    )
    label_df = pd.read_csv(
        f"label_sp3_de001_run{run_id}.csv"
    )

    label_df = label_df[
        ["id", "label", "cluster_id"]
    ].rename(columns={"id": "Train_ID"})

    foif_df = foif_df.rename(
        columns={"Score": "FOIF_Score"}
    )

    tracin_df = tracin_df.rename(
        columns={"Score": "TracIn_Score"}
    )

    density_df = (
        label_df
        .merge(loss_df, on="Train_ID", validate="one_to_one")
        .merge(foif_df, on="Train_ID", validate="one_to_one")
        .merge(tracin_df, on="Train_ID", validate="one_to_one")
    )

    density_df["Density_Group"] = density_df["cluster_id"].replace({
        "1_sparse": "Sparse",
        "0_sparse": "Sparse",
        "1_dense": "Dense",
        "0_dense": "Dense"
    })

    density_df["Sparse"] = (
        density_df["cluster_id"]
        .str.contains("sparse", case=False, na=False)
        .astype(int)
    )

    density_df["Log_Loss"] = np.log1p(
        density_df["Training_Loss"]
    )

    density_df["Log_Abs_FOIF"] = np.log1p(
        np.abs(density_df["FOIF_Score"])
    )

    density_df["Run"] = run_id

    return density_df

In [27]:
def fit_regression_for_run(density_df, run_id):
    """
    Fit signed-score and magnitude regression models for one run.
    """

    signed_model = smf.ols(
        "FOIF_Score ~ Log_Loss * Sparse",
        data=density_df
    ).fit()

    magnitude_model = smf.ols(
        "Log_Abs_FOIF ~ Log_Loss * Sparse",
        data=density_df
    ).fit()

    signed_result = {
        "Run": run_id,
        "Model": "Signed FOIF",
        "Beta_0_Intercept": signed_model.params["Intercept"],
        "Beta_1_Log_Loss": signed_model.params["Log_Loss"],
        "Beta_2_Sparse": signed_model.params["Sparse"],
        "Beta_3_Interaction": signed_model.params["Log_Loss:Sparse"],
        "Sparse_Slope": (
            signed_model.params["Log_Loss"]
            + signed_model.params["Log_Loss:Sparse"]
        ),
    }

    magnitude_result = {
        "Run": run_id,
        "Model": "Absolute FOIF magnitude",
        "Beta_0_Intercept": magnitude_model.params["Intercept"],
        "Beta_1_Log_Loss": magnitude_model.params["Log_Loss"],
        "Beta_2_Sparse": magnitude_model.params["Sparse"],
        "Beta_3_Interaction": magnitude_model.params[
            "Log_Loss:Sparse"
        ],
        "Sparse_Slope": (
            magnitude_model.params["Log_Loss"]
            + magnitude_model.params["Log_Loss:Sparse"]
        ),
    }

    return signed_result, magnitude_result

In [28]:
def calculate_decile_summary(density_df, run_id):
    """
    Calculate loss-decile statistics for one experimental run.
    """

    density_df = density_df.copy()

    density_df["Loss_Decile"] = pd.qcut(
        density_df["Training_Loss"],
        q=10,
        labels=False,
        duplicates="drop"
    ) + 1

    decile_summary = (
        density_df
        .groupby(
            ["Loss_Decile", "Sparse"],
            as_index=False
        )
        .agg(
            Count=("Train_ID", "size"),

            Mean_Loss=("Training_Loss", "mean"),
            Median_Loss=("Training_Loss", "median"),

            Mean_FOIF=("FOIF_Score", "mean"),
            FOIF_Positive_Fraction=(
                "FOIF_Score",
                lambda x: (x > 0).mean()
            ),

            Mean_TracIn=("TracIn_Score", "mean"),
            TracIn_Positive_Fraction=(
                "TracIn_Score",
                lambda x: (x > 0).mean()
            )
        )
    )

    decile_summary["Run"] = run_id

    return decile_summary

In [29]:
seeds = [1,2,3,4,5]

all_regression_results = []
all_run_data = []
all_median_loss_results = []
all_decile_results = []

for seed in seeds:
    print(f"Processing run {seed}...")

    density_df = load_and_prepare_run(
        run_id=seed,
        result_dir=""
    )

    # median_loss_run = (
    # density_df
    # .groupby("Density_Group")["Training_Loss"]
    # .median()
    # )

    # all_median_loss_results.append({
    #     "Run": seed,
    #     "Dense_Median_Loss": median_loss_run.get("Dense", np.nan),
    #     "Sparse_Median_Loss": median_loss_run.get("Sparse", np.nan)
    # })

    decile_summary_run = calculate_decile_summary(
        density_df=density_df,
        run_id=seed
    )

    all_decile_results.append(decile_summary_run)

    signed_result, magnitude_result = fit_regression_for_run(
        density_df=density_df,
        run_id=seed
    )

    all_regression_results.extend([
        signed_result,
        magnitude_result
    ])

    all_run_data.append(density_df)

regression_results_df = pd.DataFrame(
    all_regression_results
)

all_density_df = pd.concat(
    all_run_data,
    ignore_index=True
)

display(regression_results_df)

Processing run 1...
Processing run 2...
Processing run 3...
Processing run 4...
Processing run 5...


,Run,Model,Beta_0_Intercept,Beta_1_Log_Loss,Beta_2_Sparse,Beta_3_Interaction,Sparse_Slope
0,1,Signed FOIF,-0.401044,9.836590,0.585313,-10.365962,-0.529373
1,1,Absolute FOIF magnitude,-0.392500,9.632029,0.389680,-9.340002,0.292028
2,2,Signed FOIF,0.022743,-0.109287,0.154308,-0.394345,-0.503631
3,2,Absolute FOIF magnitude,0.022497,-0.107313,-0.019227,0.379461,0.272148
4,3,Signed FOIF,0.012663,0.156906,0.151768,-0.643133,-0.486227
5,3,Absolute FOIF magnitude,0.012609,0.153848,-0.011257,0.132949,0.286797
6,4,Signed FOIF,0.126178,-2.296789,0.061814,1.765674,-0.531115
7,4,Absolute FOIF magnitude,0.123557,-2.244549,-0.125414,2.528171,0.283623
8,5,Signed FOIF,0.105542,-1.890401,0.067283,1.392332,-0.498069
9,5,Absolute FOIF magnitude,0.103578,-1.851399,-0.104125,2.130490,0.279091


In [30]:
median_loss_results_df = pd.DataFrame(
    all_median_loss_results
)

display(median_loss_results_df)

""


In [31]:
coefficient_columns = [
    "Beta_0_Intercept",
    "Beta_1_Log_Loss",
    "Beta_2_Sparse",
    "Beta_3_Interaction",
    "Sparse_Slope",
]

regression_summary = (
    regression_results_df
    .groupby("Model")[coefficient_columns]
    .agg(["mean", "std"])
)

display(regression_summary)

Beta_0_Intercept           Beta_1_Log_Loss            \
                                    mean       std            mean       std   
Model                                                                          
Absolute FOIF magnitude        -0.026052  0.210548        1.116523  4.874519   
Signed FOIF                    -0.026784  0.215050        1.139404  4.978773   

                        Beta_2_Sparse           Beta_3_Interaction            \
                                 mean       std               mean       std   
Model                                                                          
Absolute FOIF magnitude      0.025932  0.209496          -0.833786  4.869598   
Signed FOIF                  0.204097  0.217661          -1.649087  4.987010   

                        Sparse_Slope            
                                mean       std  
Model                                           
Absolute FOIF magnitude     0.282737  0.007567  
Signed FOIF                -0.509683  0.019803

In [32]:
decile_results_5_runs = pd.concat(
    all_decile_results,
    ignore_index=True
)

display(decile_results_5_runs)

,Loss_Decile,Sparse,Count,Mean_Loss,Median_Loss,Mean_FOIF,FOIF_Positive_Fraction,Mean_TracIn,TracIn_Positive_Fraction,Run
0,1,0,565,0.043147,0.043176,0.015520,1.000000,0.006460,1.000000,1
1,1,1,235,0.027213,0.027956,0.028945,1.000000,0.006020,0.995745,1
2,2,0,795,0.043387,0.043384,0.015601,1.000000,0.006469,1.000000,1
3,2,1,5,0.043400,0.043395,0.047282,1.000000,0.005007,1.000000,1
4,3,0,798,0.043689,0.043665,0.018356,1.000000,0.005722,1.000000,1
...,...,...,...,...,...,...,...,...,...,...
75,6,1,546,0.084203,0.083293,0.053600,0.913919,0.005339,0.869963,5
76,7,1,800,0.208960,0.207139,0.082289,0.931250,0.004900,0.718750,5
77,8,1,800,0.418914,0.412030,0.069804,0.860000,0.003504,0.656250,5
78,9,1,800,0.759803,0.740639,-0.030251,0.395000,-0.003443,0.436250,5


In [33]:
statistics_to_summarise = [
    "Count",
    "Mean_Loss",
    "Median_Loss",
    "Mean_FOIF",
    "FOIF_Positive_Fraction",
    "Mean_TracIn",
    "TracIn_Positive_Fraction"
]

decile_summary_5_runs = (
    decile_results_5_runs
    .groupby(
        ["Loss_Decile", "Sparse"]
    )[statistics_to_summarise]
    .agg(["mean", "std"])
    .reset_index()
)

display(decile_summary_5_runs)

Loss_Decile Sparse  Count            Mean_Loss           Median_Loss  \
                        mean        std      mean       std        mean   
0            1      0  561.2  10.639549  0.043410  0.001156    0.043438   
1            1      1  239.4  11.865918  0.027143  0.001133    0.028324   
2            2      0  796.6   2.509980  0.043705  0.001173    0.043701   
3            2      1    2.8   1.643168  0.043726  0.001163    0.043723   
4            3      0  768.0  30.943497  0.044551  0.001114    0.044081   
5            3      1   32.0  30.943497  0.045821  0.001429    0.045762   
6            4      0  797.8   1.303840  0.047916  0.002736    0.047924   
7            4      1    2.6   0.547723  0.047912  0.002725    0.047911   
8            5      0  795.2   3.033150  0.048249  0.002728    0.048235   
9            5      1    4.4   3.130495  0.048327  0.002670    0.048339   
10           6      0  281.2  40.201990  0.048787  0.002800    0.048707   
11           6      1  518.8  40.201990  0.082305  0.003285    0.081823   
12           7      1  800.0   0.000000  0.196964  0.010208    0.194463   
13           8      1  800.0   0.000000  0.406039  0.016542    0.401520   
14           9      1  800.0   0.000000  0.750573  0.009535    0.737560   
15          10      1  800.0   0.000000  1.703941  0.012879    1.503071   

             Mean_FOIF           FOIF_Positive_Fraction           Mean_TracIn  \
         std      mean       std                   mean       std        mean   
0   0.001161  0.020711  0.005259               1.000000  0.000000    0.001481   
1   0.001588  0.023875  0.003686               0.963376  0.029165    0.004477   
2   0.001177  0.020824  0.005004               1.000000  0.000000    0.001467   
3   0.001168  0.041572  0.009042               1.000000  0.000000    0.004752   
4   0.001096  0.020852  0.003430               1.000000  0.000000    0.002286   
5   0.001385  0.036854  0.004448               0.958022  0.045865    0.005082   
6   0.002737  0.020177  0.003578               1.000000  0.000000    0.006755   
7   0.002686  0.044994  0.017802               1.000000  0.000000    0.004850   
8   0.002731  0.020431  0.003759               1.000000  0.000000    0.006743   
9   0.002669  0.044676  0.007168               1.000000  0.000000    0.005402   
10  0.002795  0.020333  0.004062               1.000000  0.000000    0.006773   
11  0.002529  0.055169  0.003835               0.956633  0.033712    0.005375   
12  0.010208  0.084250  0.005034               0.961250  0.023452    0.004981   
13  0.015774  0.073838  0.003259               0.871250  0.018371    0.003424   
14  0.011091 -0.021687  0.005479               0.422500  0.033506   -0.003039   
15  0.020980 -0.368642  0.008638               0.061000  0.023509   -0.019351   

             TracIn_Positive_Fraction            
         std                     mean       std  
0   0.003197                 0.795365  0.444736  
1   0.001813                 0.889636  0.230806  
2   0.003213                 0.786216  0.440520  
3   0.005978                 0.800000  0.447214  
4   0.002188                 0.795969  0.364837  
5   0.001329                 0.909930  0.182095  
6   0.002805                 1.000000  0.000000  
7   0.001173                 0.900000  0.223607  
8   0.002856                 1.000000  0.000000  
9   0.003853                 0.733333  0.434613  
10  0.002896                 1.000000  0.000000  
11  0.000963                 0.822669  0.167002  
12  0.000861                 0.715500  0.124692  
13  0.000482                 0.647750  0.056440  
14  0.000628                 0.453500  0.015522  
15  0.004578                 0.268500  0.118511

In [34]:
def mean_std_string(x):
    return f"{x.mean():.6f} ± {x.std(ddof=1):.6f}"


formatted_decile_summary = (
    decile_results_5_runs
    .groupby(
        ["Loss_Decile", "Sparse"]
    )[statistics_to_summarise]
    .agg(mean_std_string)
    .reset_index()
)

display(formatted_decile_summary)

,Loss_Decile,Sparse,Count,Mean_Loss,Median_Loss,Mean_FOIF,FOIF_Positive_Fraction,Mean_TracIn,TracIn_Positive_Fraction
0,1,0,561.200000 ± 10.639549,0.043410 ± 0.001156,0.043438 ± 0.001161,0.020711 ± 0.005259,1.000000 ± 0.000000,0.001481 ± 0.003197,0.795365 ± 0.444736
1,1,1,239.400000 ± 11.865918,0.027143 ± 0.001133,0.028324 ± 0.001588,0.023875 ± 0.003686,0.963376 ± 0.029165,0.004477 ± 0.001813,0.889636 ± 0.230806
2,2,0,796.600000 ± 2.509980,0.043705 ± 0.001173,0.043701 ± 0.001177,0.020824 ± 0.005004,1.000000 ± 0.000000,0.001467 ± 0.003213,0.786216 ± 0.440520
3,2,1,2.800000 ± 1.643168,0.043726 ± 0.001163,0.043723 ± 0.001168,0.041572 ± 0.009042,1.000000 ± 0.000000,0.004752 ± 0.005978,0.800000 ± 0.447214
4,3,0,768.000000 ± 30.943497,0.044551 ± 0.001114,0.044081 ± 0.001096,0.020852 ± 0.003430,1.000000 ± 0.000000,0.002286 ± 0.002188,0.795969 ± 0.364837
5,3,1,32.000000 ± 30.943497,0.045821 ± 0.001429,0.045762 ± 0.001385,0.036854 ± 0.004448,0.958022 ± 0.045865,0.005082 ± 0.001329,0.909930 ± 0.182095
6,4,0,797.800000 ± 1.303840,0.047916 ± 0.002736,0.047924 ± 0.002737,0.020177 ± 0.003578,1.000000 ± 0.000000,0.006755 ± 0.002805,1.000000 ± 0.000000
7,4,1,2.600000 ± 0.547723,0.047912 ± 0.002725,0.047911 ± 0.002686,0.044994 ± 0.017802,1.000000 ± 0.000000,0.004850 ± 0.001173,0.900000 ± 0.223607
8,5,0,795.200000 ± 3.033150,0.048249 ± 0.002728,0.048235 ± 0.002731,0.020431 ± 0.003759,1.000000 ± 0.000000,0.006743 ± 0.002856,1.000000 ± 0.000000
9,5,1,4.400000 ± 3.130495,0.048327 ± 0.002670,0.048339 ± 0.002669,0.044676 ± 0.007168,1.000000 ± 0.000000,0.005402 ± 0.003853,0.733333 ± 0.434613


In [35]:
# density_df["Loss_Decile"] = pd.qcut(
#     density_df["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# ) + 1

In [36]:
# decile_label_summary = (
#     density_df
#     .groupby(["Loss_Decile", "Density_Group"])
#     .agg(
#         Count=("Train_ID", "size"),
#         Mean_Loss=("Training_Loss", "mean"),

#         Mean_FOIF=("FOIF_Score", "mean"),
#         FOIF_Positive_Fraction=(
#             "FOIF_Score",
#             lambda x: (x > 0).mean()
#         ),

#         Mean_TracIn=("TracIn_Score", "mean"),
#         TracIn_Positive_Fraction=(
#             "TracIn_Score",
#             lambda x: (x > 0).mean()
#         )
#     )
#     .reset_index()
# )

# print(decile_label_summary)